# Notebook 1 — Thu thập & Tiền xử lý Dữ liệu Tài chính

| Trường | Nội dung |
|---|---|
| **Dự án** | Bayesian Uncertainty-Aware Financial Risk Forecasting |
| **Notebook** | 1 of 7 |
| **Input** | S&P 500 OHLCV từ Yahoo Finance |
| **Output** | `data/raw/sp500_clean.csv` · figures trong `reports/figures/nb1/` |
| **Trọng tâm** | Load · EDA · Quality Check · Visualization |

## 1. Import thư viện & Cấu hình đường dẫn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)

# ── Đường dẫn artifact (tương đối so với thư mục gốc project) ──────────────
FIGURES_DIR = '../reports/figures/nb1'
RAW_DIR     = '../data/raw'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RAW_DIR,     exist_ok=True)

print('✓ Imports OK')
print(f'  Figures → {FIGURES_DIR}')
print(f'  Raw data → {RAW_DIR}')

✓ Imports OK
  Figures → ../reports/figures/nb1
  Raw data → ../data/raw


## 2. Load dữ liệu S&P 500

Dataset bao gồm dữ liệu OHLCV từ **2015-01-01 đến 2024-12-31** (~2514 phiên giao dịch).

In [2]:

df = pd.read_csv( '../data/raw/sp500.csv', header=[0, 1], index_col=0 )
 

print(df.head())

# Flatten MultiIndex columns
df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
df.index   = pd.to_datetime(df.index)
df         = df.sort_index()

print(f'Shape   : {df.shape}')
print(f'Period  : {df.index.min().date()} → {df.index.max().date()}')
print(f'Columns : {df.columns.tolist()}')
print()
print(df.head())



Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2015-01-02  2058.199951  2072.360107  2046.040039  2058.899902  2708700000
2015-01-05  2020.579956  2054.439941  2017.339966  2054.439941  3799120000
2015-01-06  2002.609985  2030.250000  1992.439941  2022.150024  4460110000
2015-01-07  2025.900024  2029.609985  2005.550049  2005.550049  3805480000
2015-01-08  2062.139893  2064.080078  2030.609985  2030.609985  3934010000
Shape   : (2514, 5)
Period  : 2015-01-02 → 2024-12-27
Columns : ['Close', 'High', 'Low', 'Open', 'Volume']

                  Close         High          Low         Open      Volume
Date                                                                      
2015-01-02  2058.199951  2072.360107  2046.040039  2058.899902  2708700000
2015-01-05  2020.579956  2054.439941  2017.339966  2054.439941  379

## 3. Thống kê mô tả

In [3]:
desc = df.describe()
print('=== Thống kê mô tả ===')
print(desc.round(2).to_string())

=== Thống kê mô tả ===
         Close     High      Low     Open        Volume
count  2514.00  2514.00  2514.00  2514.00  2.514000e+03
mean   3354.11  3371.41  3334.46  3353.62  4.006123e+09
std    1081.96  1087.18  1076.24  1081.73  9.574610e+08
min    1829.08  1847.00  1810.10  1833.40  0.000000e+00
25%    2431.94  2441.42  2420.15  2431.92  3.427938e+09
50%    3004.28  3016.19  2989.73  3004.17  3.820510e+09
75%    4203.59  4232.43  4182.48  4205.65  4.341800e+09
max    6090.27  6099.97  6079.98  6089.03  9.976520e+09


## 4. Kiểm tra chất lượng dữ liệu

In [4]:
# ── Missing values ─────────────────────────────────────────────────────────
missing      = df.isnull().sum()
missing_pct  = (missing / len(df) * 100).round(2)

print('=== Missing Values ===')
for col in df.columns:
    print(f'  {col:<10}: {missing[col]}  ({missing_pct[col]}%)')

# ── Kiểm tra business days bị thiếu ────────────────────────────────────────
bdays        = pd.bdate_range(start=df.index.min(), end=df.index.max())
missing_days = bdays.difference(df.index)

print(f'\nExpected business days : {len(bdays)}')
print(f'Actual trading days    : {len(df)}')
print(f'Missing business days  : {len(missing_days)}  (holidays / market closures)')

# ── Kiểm tra Inf ────────────────────────────────────────────────────────────
inf_count = np.isinf(df.select_dtypes('number')).sum().sum()
print(f'\nInfinite values        : {inf_count}')

# ── Kiểm tra thứ tự thời gian ───────────────────────────────────────────────
is_sorted = df.index.is_monotonic_increasing
print(f'Chronologically sorted : {is_sorted}')

=== Missing Values ===
  Close     : 0  (0.0%)
  High      : 0  (0.0%)
  Low       : 0  (0.0%)
  Open      : 0  (0.0%)
  Volume    : 0  (0.0%)

Expected business days : 2606
Actual trading days    : 2514
Missing business days  : 92  (holidays / market closures)

Infinite values        : 0
Chronologically sorted : True


## 5. Visualization — Tổng quan thị trường

In [ ]:
# ── Plot 1: Close Price ─────────────────────────────────────────────────────
daily_ret = np.log(df['Close'] / df['Close'].shift(1)).dropna()  # tính sớm để dùng ở cells sau

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['Close'], linewidth=1.2, color='steelblue')
ax.set_title('S&P 500 — Close Price (2015–2024)', fontsize=13)
ax.set_ylabel('Price (USD)')
ax.grid(alpha=0.3)

# Annotate COVID crash
covid_date = pd.Timestamp('2020-03-23')
if covid_date in df.index:
    ax.axvline(covid_date, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.annotate('COVID crash\n2020-03', xy=(covid_date, df.loc[covid_date, 'Close']),
                xytext=(30, 40), textcoords='offset points',
                fontsize=8, color='red',
                arrowprops=dict(arrowstyle='->', color='red', lw=1))

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/close_price.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/close_price.png')

In [ ]:
# ── Plot 2: Trading Volume ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df.index, df['Volume'], width=1.5, color='coral', alpha=0.7)
ax.set_title('S&P 500 — Trading Volume (2015–2024)', fontsize=13)
ax.set_ylabel('Volume')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/trading_volume.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/trading_volume.png')

In [ ]:
# ── Plot 3: Daily Log Returns ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_ret.index, daily_ret, linewidth=0.6, color='seagreen', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('S&P 500 — Daily Log Return (2015–2024)', fontsize=13)
ax.set_ylabel('Log Return')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/daily_returns.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/daily_returns.png')

In [ ]:
# ── Phân phối Log Return — Histogram + KDE ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(daily_ret, bins=80, kde=True, ax=ax, color='steelblue')
ax.set_title('Phân phối Log Return S&P 500 — Histogram + KDE', fontsize=13)
ax.set_xlabel('Log Return')
ax.set_ylabel('Count')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/return_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/return_histogram.png')

In [ ]:
# ── Phân phối Log Return — Boxplot ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x=daily_ret, ax=ax, color='coral')
ax.set_title('Phân phối Log Return S&P 500 — Boxplot', fontsize=13)
ax.set_xlabel('Log Return')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/return_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/return_boxplot.png')

## 6. Làm sạch & Lưu dữ liệu

In [7]:
df_clean = df.copy()

# Forward-fill khoảng trống nhỏ (nếu có)
df_clean = df_clean.ffill()

# Lưu
output_path = '../data/raw/sp500_clean.csv'
df_clean.to_csv(output_path)

print(f'✓ Saved → {output_path}')
print(f'  Shape  : {df_clean.shape}')
print(f'  Period : {df_clean.index.min().date()} → {df_clean.index.max().date()}')
print(f'  Columns: {df_clean.columns.tolist()}')
print()
print('Nhận xét:')
print(f'  - Dataset sạch, không có missing values trong OHLCV')
print(f'  - {len(missing_days)} ngày lễ/thị trường đóng cửa — bình thường với trading data')
print(f'  - Dữ liệu có tính non-stationary → cần dùng return thay vì raw price')

✓ Saved → ../data/raw/sp500_clean.csv
  Shape  : (2514, 5)
  Period : 2015-01-02 → 2024-12-27
  Columns: ['Close', 'High', 'Low', 'Open', 'Volume']

Nhận xét:
  - Dataset sạch, không có missing values trong OHLCV
  - 92 ngày lễ/thị trường đóng cửa — bình thường với trading data
  - Tỷ lệ Strong Movement (|return|>1%): 24.9%
  - Dữ liệu có tính non-stationary → cần dùng return thay vì raw price
